# Notebook 2: Hybrid RAG with Atlas Search, Vector Search, and Manual RRF

This notebook demonstrates:

- **Voyage AI** for embeddings
- **MongoDB Atlas `$search`** for keyword retrieval
- **MongoDB Atlas `$vectorSearch`** for semantic retrieval
- **Manual Reciprocal Rank Fusion (RRF)** in Python
- **Voyage AI `rerank-2.5`** for final re-ranking

> It is safe to run in GitHub Codespaces. If `VOYAGE_API_KEY` or `MONGODB_URI` is missing, external calls are skipped and the notebook still completes without errors.

In [ ]:
%pip install -q "voyageai>=0.3.2" "pymongo>=4.6.0" "langchain-mongodb>=0.2.0" "python-dotenv>=1.0.0" "ipykernel>=6.0.0"

In [11]:
import os
from dataclasses import dataclass
from typing import List

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


class VoyageAIEmbeddings(Embeddings):
    """LangChain-compatible embeddings wrapper for Voyage AI."""

    def __init__(self, api_key: str | None, model: str = "voyage-3"):
        self.api_key = api_key
        self.model = model
        self.client = None
        if api_key:
            import voyageai

            self.client = voyageai.Client(api_key=api_key)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        if not self.client:
            raise ValueError("Voyage client is not initialized.")
        result = self.client.embed(texts, model=self.model, input_type="document")
        return result.embeddings

    def embed_query(self, text: str) -> List[float]:
        if not self.client:
            raise ValueError("Voyage client is not initialized.")
        result = self.client.embed([text], model=self.model, input_type="query")
        return result.embeddings[0]


VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")
MONGODB_URI = os.getenv("MONGODB_URI")
EMBED_MODEL = "voyage-3"
RERANK_MODEL = "rerank-2.5"
DB_NAME = "pyconlt2026"
COLLECTION_NAME = "manual_rrf_demo"
VECTOR_INDEX_NAME = "vector_search_index"
TEXT_INDEX_NAME = "text_search_index"

embeddings = VoyageAIEmbeddings(VOYAGE_API_KEY, model=EMBED_MODEL)

print("VOYAGE_API_KEY set:", bool(VOYAGE_API_KEY))
print("MONGODB_URI set   :", bool(MONGODB_URI))

VOYAGE_API_KEY set: False
MONGODB_URI set   : False


/Users/piti.champeethong/Downloads/github/20260410_PyConLT2026/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [12]:
@dataclass
class DemoDoc:
    doc_id: str
    content: str

documents = [
    DemoDoc("doc-01", "Reciprocal Rank Fusion combines ranked lists without comparing raw scores."),
    DemoDoc("doc-02", "Relative Score Fusion works best when scores are normalized and calibrated."),
    DemoDoc("doc-03", "BM25 is a strong keyword retrieval method for exact terms, acronyms, and product codes."),
    DemoDoc("doc-04", "Dense vector retrieval finds semantically similar documents even when wording differs."),
    DemoDoc("doc-05", "Cross-encoder re-ranking scores each query-document pair directly."),
    DemoDoc("doc-06", "Hybrid search combines keyword retrieval and vector retrieval to improve recall and precision."),
    DemoDoc("doc-07", "RAG pipelines reduce hallucinations by grounding generation in retrieved context."),
    DemoDoc("doc-08", "Chunking strategy affects retrieval quality and downstream answer quality."),
    DemoDoc("doc-09", "Voyage AI provides embedding models such as voyage-3 for retrieval tasks."),
    DemoDoc("doc-10", "MongoDB Atlas Search supports full-text search and vector search in one platform."),
    DemoDoc("doc-11", "MongoDB `$rankFusion` applies native Reciprocal Rank Fusion across multiple pipelines."),
    DemoDoc("doc-12", "Re-ranking improves the final ordering after retrieval and fusion."),
]

print(f"Loaded {len(documents)} demo documents")
for doc in documents:
    print(f"- {doc.doc_id}: {doc.content}")

Loaded 12 demo documents
- doc-01: Reciprocal Rank Fusion combines ranked lists without comparing raw scores.
- doc-02: Relative Score Fusion works best when scores are normalized and calibrated.
- doc-03: BM25 is a strong keyword retrieval method for exact terms, acronyms, and product codes.
- doc-04: Dense vector retrieval finds semantically similar documents even when wording differs.
- doc-05: Cross-encoder re-ranking scores each query-document pair directly.
- doc-06: Hybrid search combines keyword retrieval and vector retrieval to improve recall and precision.
- doc-07: RAG pipelines reduce hallucinations by grounding generation in retrieved context.
- doc-08: Chunking strategy affects retrieval quality and downstream answer quality.
- doc-09: Voyage AI provides embedding models such as voyage-3 for retrieval tasks.
- doc-10: MongoDB Atlas Search supports full-text search and vector search in one platform.
- doc-11: MongoDB `$rankFusion` applies native Reciprocal Rank Fusion acro

## Step 1: Generate embeddings with Voyage AI

If credentials are missing, the cell prints a message and skips the live API call.

In [13]:
lc_documents = [
    Document(page_content=doc.content, metadata={"doc_id": doc.doc_id})
    for doc in documents
]

print(f"Prepared {len(lc_documents)} LangChain documents")
for doc in lc_documents:
    print(f"- {doc.metadata['doc_id']}: {doc.page_content}")

Prepared 12 LangChain documents
- doc-01: Reciprocal Rank Fusion combines ranked lists without comparing raw scores.
- doc-02: Relative Score Fusion works best when scores are normalized and calibrated.
- doc-03: BM25 is a strong keyword retrieval method for exact terms, acronyms, and product codes.
- doc-04: Dense vector retrieval finds semantically similar documents even when wording differs.
- doc-05: Cross-encoder re-ranking scores each query-document pair directly.
- doc-06: Hybrid search combines keyword retrieval and vector retrieval to improve recall and precision.
- doc-07: RAG pipelines reduce hallucinations by grounding generation in retrieved context.
- doc-08: Chunking strategy affects retrieval quality and downstream answer quality.
- doc-09: Voyage AI provides embedding models such as voyage-3 for retrieval tasks.
- doc-10: MongoDB Atlas Search supports full-text search and vector search in one platform.
- doc-11: MongoDB `$rankFusion` applies native Reciprocal Rank Fusi

In [14]:
collection = None
vector_store = None

if not MONGODB_URI:
    print("Skipping MongoDB setup: MONGODB_URI is not set.")
elif not VOYAGE_API_KEY:
    print("Skipping MongoDB setup: VOYAGE_API_KEY is not set.")
else:
    from pymongo import MongoClient
    from langchain_mongodb import MongoDBAtlasVectorSearch

    client = MongoClient(MONGODB_URI)
    collection = client[DB_NAME][COLLECTION_NAME]
    collection.drop()

    vector_store = MongoDBAtlasVectorSearch(
        collection=collection,
        embedding=embeddings,
        index_name=VECTOR_INDEX_NAME,
        text_key="content",
        embedding_key="embedding",
        relevance_score_fn="cosine",
    )
    vector_store.add_documents(lc_documents)
    print(f"Inserted {len(lc_documents)} LangChain documents into {DB_NAME}.{COLLECTION_NAME}")

Skipping MongoDB setup: MONGODB_URI is not set.


## Step 2: Create Atlas Search indexes

- Use **LangChain** to create/update the vector index
- Use **PyMongo** to create the Atlas full-text search index (`$search`)

In [15]:
if collection is None or vector_store is None:
    print("Skipping index creation: MongoDB collection is not available.")
else:
    from pymongo.operations import SearchIndexModel

    # Vector index via LangChain API
    try:
        vector_store.create_vector_search_index(dimensions=1024, update=True)
        print(f"Requested create/update for {VECTOR_INDEX_NAME} via LangChain")
    except Exception as e:
        print(f"Vector index note: {e}")

    # Text index via PyMongo for keyword retrieval
    try:
        existing = {idx["name"] for idx in collection.list_search_indexes()}
    except Exception:
        existing = set()

    if TEXT_INDEX_NAME not in existing:
        collection.create_search_index(SearchIndexModel(
            definition={
                "mappings": {
                    "dynamic": False,
                    "fields": {
                        "content": {"type": "string", "analyzer": "lucene.standard"}
                    }
                }
            },
            name=TEXT_INDEX_NAME,
            type="search",
        ))
        print(f"Requested creation of {TEXT_INDEX_NAME}")
    else:
        print(f"{TEXT_INDEX_NAME} already exists")

Skipping index creation: MongoDB collection is not available.


## Step 3: Keyword search with Atlas `$search`

Run the keyword retriever first.

In [16]:
query = "Which algorithm merges ranked lists without comparing raw scores?"
keyword_results = []

if collection is None:
    print("Skipping $search query: MongoDB collection is not available.")
else:
    keyword_results = list(collection.aggregate([
        {
            "$search": {
                "index": TEXT_INDEX_NAME,
                "text": {"query": query, "path": "content"},
            }
        },
        {"$limit": 10},
        {"$project": {"_id": 0, "doc_id": 1, "content": 1}},
    ]))

    print(f"Keyword results: {len(keyword_results)}")
    for item in keyword_results:
        print(item)

Skipping $search query: MongoDB collection is not available.


## Step 4: Vector retrieval with LangChain `MongoDBAtlasVectorSearch`

Run semantic retrieval using LangChain vector store APIs (`similarity_search_with_score`).

In [17]:
vector_results = []

if vector_store is None or not VOYAGE_API_KEY:
    print("Skipping vector similarity search: credentials or data are missing.")
else:
    results_with_scores = vector_store.similarity_search_with_score(query, k=10)

    for i, (doc, score) in enumerate(results_with_scores, start=1):
        raw_id = doc.metadata.get("doc_id") or doc.metadata.get("_id") or f"vec-{i}"
        vector_results.append({
            "doc_id": str(raw_id),
            "content": doc.page_content,
            "score": float(score),
        })

    print(f"Vector results: {len(vector_results)}")
    for item in vector_results:
        print(item)

Skipping vector similarity search: credentials or data are missing.


## Step 5: Manual RRF fusion

Reciprocal Rank Fusion combines both result lists using rank positions only:

$$
RRF(d) = \sum_{r \in R} \frac{1}{k + rank_r(d)}
$$

This is simple, stable, and works well when keyword and vector scores are not directly comparable.

In [18]:
def reciprocal_rank_fusion(result_lists, k=60):
    scores = {}
    for ranked_list in result_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            doc_id = doc["doc_id"]
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return scores

fused_scores = reciprocal_rank_fusion([keyword_results, vector_results])
doc_pool = {item["doc_id"]: item for item in keyword_results + vector_results}
fused_docs = sorted(
    doc_pool.values(),
    key=lambda item: fused_scores.get(item["doc_id"], 0.0),
    reverse=True,
)[:10]

print(f"Fused results: {len(fused_docs)}")
for item in fused_docs:
    print(item["doc_id"], round(fused_scores[item["doc_id"]], 5), item["content"])

Fused results: 0


## Step 6: Re-rank with a cross-encoder

A cross-encoder reads the query and document together and produces one relevance score:

$$
score = CrossEncoder(query, document)
$$

This is slower than retrieval, but more precise.

In [19]:
if not VOYAGE_API_KEY:
    print("Skipping re-ranking: VOYAGE_API_KEY is not set.")
elif not fused_docs:
    print("Skipping re-ranking: no fused documents available.")
else:
    import voyageai

    vo = voyageai.Client(api_key=VOYAGE_API_KEY)
    rerank_result = vo.rerank(
        query=query,
        documents=[item["content"] for item in fused_docs],
        model=RERANK_MODEL,
        top_k=3,
    )

    print("Top 3 after re-ranking:")
    for rank, item in enumerate(rerank_result.results, start=1):
        doc = fused_docs[item.index]
        print(f"{rank}. {doc['doc_id']} | relevance={item.relevance_score:.4f}")
        print(f"   {doc['content']}")

Skipping re-ranking: VOYAGE_API_KEY is not set.
